# 03 - Feature engineering

Every candidate feature listed in the brief is checked for (a) a defensible definition,
(b) availability at prediction time and (c) whether it simply duplicates information already present.

In [1]:
import sys, json, warnings
from pathlib import Path
ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
from IPython.display import Image, display
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 40); pd.set_option("display.max_colwidth", 80)
from src import config as C
RES, FIG = C.Paths().results, C.Paths().figures
def fig(name, width=900):
    display(Image(filename=str(FIG / name), width=width))
from src import feature_engineering as FE
clean = pd.read_csv(C.Paths().processed / "clean_data.csv")
FE.candidate_feature_review(clean)

,feature,mean,std,coef_of_variation,n_unique,spearman_with_distance,decision,reason
0,total_household_vehicles,1.772,1.079,0.609,8,-0.126,rejected,exact sum of Car_Ownership and TwoW_Ownership
1,vehicle_ownership_category,2.147,1.115,0.519,4,-0.128,used,captures the none / 2W only / car only / both pattern
2,workers_share,0.401,0.209,0.521,20,-0.017,used,bounded share of working members (capped at HH_Size for 3 records)
3,dependency_ratio,2.156,1.485,0.688,20,0.017,rejected,"monotone transform of workers_share, so it duplicates it"
4,available_mode_count,3.608,1.101,0.305,6,-0.420,rejected,linear sum of the availability indicators already used
5,public_transport_available,1.000,0.000,0.000,1,NaN,rejected,constant: suburban rail is available to everyone
6,private_vehicle_available,0.840,0.367,0.437,2,-0.140,used,not a linear combination (logical OR of two indicators)
7,time_advantage_bus_vs_car,-5.624,6.107,-1.086,194,-0.999,rejected,fixed multiple of distance because times are computed from it
8,time_advantage_train_vs_car,7.111,7.708,1.084,222,0.999,rejected,fixed multiple of distance because times are computed from it
9,cost_advantage_bus_vs_car,17.348,26.991,1.556,81,0.997,rejected,near-deterministic function of distance


Most "relative advantage" features fail check (c). Because times and costs are computed from distance,
the bus-versus-car time difference is just distance multiplied by a constant (Spearman rho = -0.999
with distance), and cost per km is constant apart from rounding. Total vehicles and the count of
available modes are exact sums of columns already in the model. Public-transport availability is
constant because rail is available to everyone.

Five derived variables are kept: log distance, vehicle-ownership category, private vehicle
available, workers share (workers per household member, capped at 1) and a has-children flag.
Rare occupation codes 1 and 4 (7 people) are pooled for estimation.

In [2]:
print("categorical:", FE.CATEGORICAL)
print("numeric:", FE.NUMERIC)
FE.FeatureBuilder().transform(clean).head()

categorical: ['Trip_Purpose', 'Income', 'Age', 'Education', 'occ_group', 'Dwelling', 'Gender', 'ParkingSpace', 'Area_Type', 'SAPT', 'veh_own_cat']
numeric: ['Distance_km', 'log_distance', 'walk_time_min', 'TwoW_time_min', 'auto_time_min', 'car_time_min', 'bus_time_min', 'local_train_time_min', 'TC_2W', 'TC_Auto', 'TC_Car', 'TC_Bus', 'TC_LocalTrain', 'HH_Size', 'Children', 'Working_Adults', 'workers_share', 'has_children', 'Car_Ownership', 'TwoW_Ownership', 'private_av', 'Walk_AV', 'TwoW_AV', 'Auto_AV', 'Car_AV', 'Bus_AV']


,Trip_Purpose,Income,Age,Education,occ_group,Dwelling,Gender,ParkingSpace,Area_Type,SAPT,veh_own_cat,Distance_km,log_distance,walk_time_min,TwoW_time_min,auto_time_min,car_time_min,bus_time_min,local_train_time_min,TC_2W,TC_Auto,TC_Car,TC_Bus,TC_LocalTrain,HH_Size,Children,Working_Adults,workers_share,has_children,Car_Ownership,TwoW_Ownership,private_av,Walk_AV,TwoW_AV,Auto_AV,Car_AV,Bus_AV
0,2,1,2,2,3,4,1,0,0,1,0,2.5,0.916291,37.5,6.0,7.5,6.8,8.8,4.3,4,24,11,7,5,6,2,1,0.166667,1,0,0,0,0,0,1,0,0
1,2,1,1,2,5,4,1,0,0,1,3,2.5,0.916291,37.5,6.0,7.5,6.8,8.8,4.3,4,24,11,7,5,2,0,1,0.500000,0,1,1,1,0,1,0,0,0
2,2,3,2,1,3,1,1,0,1,1,3,20.5,3.020425,306.9,49.1,61.4,55.8,72.2,35.1,33,201,91,30,10,6,1,1,0.166667,1,1,1,1,0,1,1,1,0
3,2,1,2,2,3,4,1,0,0,1,3,19.3,2.960105,288.8,46.2,57.8,52.5,67.9,33.0,31,189,86,29,5,4,2,1,0.250000,1,1,1,1,0,1,0,1,0
4,2,1,2,2,5,4,1,0,0,1,1,1.2,0.182322,18.2,2.9,3.6,3.3,4.3,2.1,2,15,5,7,5,5,3,1,0.200000,1,0,1,1,1,1,1,0,0


The features are built inside the model pipeline (`FeatureBuilder`), not stored in the data. A policy
scenario that changes a raw time or cost therefore updates every derived feature automatically, and
nothing derived from the target or from other records can leak into a prediction.

For the discrete choice models the same data are reshaped into person-alternative long format, one
row per available alternative:

In [3]:
long = pd.read_csv(C.Paths().processed / "train_long_format.csv")
print(long.shape, "rows for", long["ID"].nunique(), "people")
long.head(12)

(1520, 14) rows for 424 people


,ID,alternative,alt_code,chosen,time_min,cost_inr,Distance_km,Income,Age,Trip_Purpose,SAPT,ParkingSpace,Car_Ownership,TwoW_Ownership
0,620,twow,2,0,26.2,18.0,10.9,1,2,1,1,1,1,1
1,620,car,4,0,29.7,49.0,10.9,1,2,1,1,1,1,1
2,620,train,6,1,18.7,5.0,10.9,1,2,1,1,1,1,1
3,590,twow,2,0,27.4,18.0,11.4,1,2,2,1,1,0,1
4,590,train,6,1,19.5,5.0,11.4,1,2,2,1,1,0,1
5,21,walk,1,0,213.0,0.0,14.2,1,3,2,1,0,0,0
6,21,train,6,1,24.3,5.0,14.2,1,3,2,1,0,0,0
7,10,auto,3,1,7.5,24.0,2.5,1,2,2,1,0,0,0
8,10,train,6,0,4.3,5.0,2.5,1,2,2,1,0,0,0
9,167,walk,1,0,20.0,0.0,1.3,2,2,2,1,1,1,1
